# CartPole: A Top-Level Overview

CartPole (also known as the inverted pendulum) is one of the most classic environments in Reinforcement Learning.

### The Problem
Imagine balancing a broomstick on your hand. You have to constantly move your hand left or right to keep the stick from falling over. This is exactly what the CartPole environment simulates.

*   **The Agent:** The cart moving along a frictionless track.
*   **The Objective:** Keep the pole balanced upright for as long as possible.
*   **The Actions:** The agent can apply a force of +1 (push right) or -1 (push left) to the cart.
*   **The Observations (State):** The agent knows 4 things at any given time:
    1.  Cart Position
    2.  Cart Velocity
    3.  Pole Angle
    4.  Pole Angular Velocity
*   **The Reward:** The agent receives a reward of +1 for every timestep the pole remains upright. The episode ends if the pole falls past a certain angle or the cart moves off-screen.

In [ ]:

# CartPole hello world for Colab — Python 3.12 safe version
# Fixes the gym 0.21 / pkgutil.ImpImporter install error

# --- Install: pin recent versions, avoid legacy gym ---
!pip install -q --upgrade pip
!pip install -q "stable-baselines3>=2.3.0" "gymnasium>=0.29" imageio imageio-ffmpeg

# --- Imports ---
import gymnasium as gym
import numpy as np
import imageio
from stable_baselines3 import PPO
from IPython.display import HTML
from base64 import b64encode

# --- 1. Train ---
train_env = gym.make("CartPole-v1")
model = PPO("MlpPolicy", train_env, verbose=0, seed=0)
print("Training PPO for 25k steps (~30s on Colab CPU)...")
model.learn(total_timesteps=25_000)
print("Done.")

# --- 2. Roll out and capture frames ---
render_env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = render_env.reset(seed=42)
frames, total_reward = [], 0
for _ in range(500):
    frames.append(render_env.render())
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = render_env.step(action)
    total_reward += reward
    if terminated or truncated:
        break
render_env.close()
print(f"Episode length: {len(frames)} steps · total reward: {total_reward}")

# --- 3. Encode and display inline ---
video_path = "/tmp/cartpole.mp4"
imageio.mimsave(video_path, frames, fps=30, codec="libx264")

with open(video_path, "rb") as f:
    b64 = b64encode(f.read()).decode()
HTML(f"""
<video width="500" controls autoplay loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
""")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.8 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training PPO for 25k steps (~30s on Colab CPU)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Done.


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

Episode length: 500 steps · total reward: 500.0


### Lower-Level Code Explanation (Line-by-Line Breakdown)

Here is a detailed explanation of the code cell above:

*   **Lines 5-6 (Installation):** Installs the necessary libraries. We upgrade `pip` and install `stable-baselines3` (which contains our RL algorithm, PPO), `gymnasium` (which contains the CartPole environment), and `imageio` (for saving the video).
*   **Lines 9-14 (Imports):** Imports the modules we just installed, along with `IPython.display.HTML` and `base64` to securely embed the resulting video into the Colab notebook.
*   **Lines 17-21 (Training Phase):**
    *   `gym.make("CartPole-v1")` creates the standard training environment.
    *   `PPO("MlpPolicy", ...)` initializes the Proximal Policy Optimization (PPO) agent. It uses a Multi-Layer Perceptron (MLP) neural network to map observations to actions.
    *   `model.learn(total_timesteps=25_000)` executes the training loop. The agent plays the game over and over for 25,000 steps, updating its neural network to maximize rewards.
*   **Lines 24-35 (Rollout and Capture Phase):**
    *   We create a *new* environment, but this time with `render_mode="rgb_array"` so we can capture image frames of the agent playing.
    *   We loop for a maximum of 500 steps. In each step:
        1.  `render_env.render()` captures the current screen (frame).
        2.  `model.predict(obs)` asks our *trained* model to decide the best action based on the current state (`obs`).
        3.  `render_env.step(action)` executes that action in the environment.
    *   The loop breaks early if the agent drops the pole (`terminated`) or runs out of time (`truncated`).
*   **Lines 38-48 (Video Encoding and Display):**
    *   `imageio.mimsave(...)` takes our list of captured `frames` and compiles them into an MP4 file at 30 frames-per-second.
    *   We read that MP4 file, convert it to a base64 string, and inject it directly into standard HTML `<video>` tags so you can watch the agent succeed directly in the cell output!